# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kailaswadje/FlyRank-Internship_ML_Assignment_01_Week_01/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **⚠️ Execution note:** same as `w03_data_contract.ipynb` — this queries the gated Hugging Face warehouse using your own `HF_TOKEN`. It hasn't been run yet in this file. The queries below use the real, confirmed `fact_content_daily_performance` schema from your last run (no guessed column names this time) — run it top-to-bottom in Colab before committing.

## 1. Build the feature vector

Same slice as the data contract: Lane 4, `month = '2026-03'`, visible pages only (`impressions_90d`-equivalent: `gsc_impressions >= 500` for the month). Feature vector below is built entirely from confirmed real columns in `fact_content_daily_performance` — no `dim_content` join yet, since I haven't verified that table's schema the same way (see Section 4 for why I'm holding off rather than guessing).

**Engineered features, with categorical handling:** all channel-share features are derived ratios (numeric), and I engineer one genuine categorical feature — `primary_channel_mar` — by taking the argmax across the six channel-session columns already in the table (organic / direct / referral / social / paid / AI), rather than guessing at a `content_type` column I haven't confirmed exists.

In [1]:
# %pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':   f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

Paste your Hugging Face READ token (hf_...): ··········


In [2]:
raw = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        AVG(gsc_avg_position)                                       AS avg_position_mar,
        SUM(gsc_impressions)                                        AS impressions_mar,
        SUM(gsc_clicks)                                             AS clicks_mar,
        SUM(ga4_sessions)                                           AS sessions_mar,
        SUM(ga4_engaged_sessions)                                   AS engaged_sessions_mar,
        SUM(ga4_total_engagement_sec)                               AS engagement_sec_mar,
        SUM(scroll_events)                                          AS scroll_events_mar,
        SUM(sessions_organic)  AS sess_organic,  SUM(sessions_direct)   AS sess_direct,
        SUM(sessions_referral) AS sess_referral, SUM(sessions_social)   AS sess_social,
        SUM(sessions_paid)     AS sess_paid,     SUM(sessions_ai)       AS sess_ai
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 500
""").df()

import pandas as pd
import numpy as np

# Derived numeric features (ratios) -- NULLIF already guards the SQL side; guard again here
# for any all-zero denominator that survives to pandas.
raw['ctr_mar'] = raw['clicks_mar'] / raw['impressions_mar'].replace(0, np.nan)
raw['engagement_rate_mar'] = raw['engaged_sessions_mar'] / raw['sessions_mar'].replace(0, np.nan)
raw['scroll_rate_mar'] = raw['scroll_events_mar'] / raw['sessions_mar'].replace(0, np.nan)
raw['engagement_sec_per_session_mar'] = raw['engagement_sec_mar'] / raw['sessions_mar'].replace(0, np.nan)

# Engineered categorical feature: primary_channel_mar (argmax across channel session columns)
channel_cols = ['sess_organic', 'sess_direct', 'sess_referral', 'sess_social', 'sess_paid', 'sess_ai']
raw['primary_channel_mar'] = raw[channel_cols].idxmax(axis=1).str.replace('sess_', '', regex=False)
# When every channel is 0 (no sessions at all that month), idxmax still picks one arbitrarily --
# mark those explicitly as 'none' rather than silently mislabeling them.
raw.loc[raw[channel_cols].sum(axis=1) == 0, 'primary_channel_mar'] = 'none'

# Fills: ratio features get NaN when their denominator is 0 (no impressions/sessions that month).
# Filling with 0 would claim "measured and zero" when the truth is "not enough activity to measure" --
# so I keep these as NaN and drop them at modeling time (dropna), rather than silently filling.
feature_cols = ['avg_position_mar', 'impressions_mar', 'ctr_mar', 'sessions_mar',
                'engagement_rate_mar', 'scroll_rate_mar', 'engagement_sec_per_session_mar',
                'primary_channel_mar']

print(f'feature vector shape: {raw.shape}')
print(f'rows with at least one NaN ratio feature: {raw[feature_cols].isna().any(axis=1).sum()}')
raw[['client_hash_id', 'content_hash_id'] + feature_cols].head(8)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feature vector shape: (61924, 20)
rows with at least one NaN ratio feature: 21311


/tmp/ipykernel_2469/1904462976.py:33: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  raw['primary_channel_mar'] = raw[channel_cols].idxmax(axis=1).str.replace('sess_', '', regex=False)


,client_hash_id,content_hash_id,avg_position_mar,impressions_mar,ctr_mar,sessions_mar,engagement_rate_mar,scroll_rate_mar,engagement_sec_per_session_mar,primary_channel_mar
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.209549,6523.0,0.001073,1.0,0.0,0.000000,0.000000,organic
1,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.724039,5630.0,0.001066,3.0,0.0,0.000000,0.000000,organic
2,client_73cda7b4e4f265ea,content_a7da352b73b02668,7.244844,4944.0,0.002629,2.0,0.0,0.000000,0.000000,organic
3,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,5.258331,7709.0,0.002594,12.0,0.0,0.083333,0.000000,organic
4,client_73cda7b4e4f265ea,content_20403327d8d9374c,8.834415,3561.0,0.002808,22.0,0.0,0.045455,0.090909,paid
5,client_73cda7b4e4f265ea,content_f8df6b20d18c4374,6.638543,798.0,0.001253,0.0,NaN,NaN,NaN,none
6,client_73cda7b4e4f265ea,content_419dc7d89aee9698,5.531187,1868.0,0.003212,18.0,0.0,0.000000,0.000000,paid
7,client_73cda7b4e4f265ea,content_30fc0ffeed8d67e6,9.745353,2481.0,0.007255,3.0,0.0,0.000000,0.000000,paid


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `avg_position_mar` | Average GSC search position over March | Never missing if a row exists (aggregated from a required numeric column) | Knowable at month's end |
| `impressions_mar` | Total GSC impressions over March | Same as above | Knowable at month's end |
| `ctr_mar` | Clicks ÷ impressions over March | NaN when `impressions_mar` is 0 — kept as NaN, not filled, since "no impressions" isn't the same claim as "0% CTR" | Knowable at month's end |
| `sessions_mar` | Total GA4 sessions over March | Never missing if a row exists | Knowable at month's end |
| `engagement_rate_mar` | Engaged sessions ÷ total sessions | NaN when `sessions_mar` is 0, kept as NaN for the same reason as CTR | Knowable at month's end |
| `scroll_rate_mar` | Scroll events ÷ total sessions | NaN when `sessions_mar` is 0 | Knowable at month's end |
| `engagement_sec_per_session_mar` | Average engaged time per session | NaN when `sessions_mar` is 0 | Knowable at month's end |
| `primary_channel_mar` (categorical) | Which acquisition channel (organic/direct/referral/social/paid/AI) brought the most sessions that month | Explicitly labeled `'none'` when all six channel columns are 0, rather than defaulting to whichever column happens to sort first | Knowable at month's end |

Every feature here is a **closed-month aggregate** — nothing reaches past `month = '2026-03'`, so nothing here can see the future relative to any label I'd define on a later window.

In [3]:
# No additional query needed here -- the table above documents each feature already
# built and printed in Section 1.

## 3. The leakage hunt

Attacking my own feature set with three checks, not just describing them:

**Attack 1 — tautological leakage.** If my eventual label is something like `is_low_ctr` (CTR below the month's median, same as the data-contract notebook), then `ctr_mar` is not a real feature for that label — it's arithmetically almost the label itself. Below I quantify exactly how close, so this isn't just an assertion.

In [4]:
raw['is_low_ctr'] = (raw['ctr_mar'] < raw['ctr_mar'].median()).astype(int)

# How separable is is_low_ctr using ONLY ctr_mar? If this is near-perfect, ctr_mar
# is not a feature for this label -- it IS the label, restated.
from sklearn.metrics import roc_auc_score
valid = raw.dropna(subset=['ctr_mar', 'is_low_ctr'])
# ctr_mar predicts is_low_ctr almost by definition -- using it "as a score" directly:
naive_auc = roc_auc_score(valid['is_low_ctr'], -valid['ctr_mar'])  # lower ctr -> higher risk
print(f"AUC using ctr_mar ALONE to predict is_low_ctr: {naive_auc:.3f}")
print("-> this is expected to sit very close to 1.0, because is_low_ctr was DEFINED from ctr_mar.")
print("   Conclusion: ctr_mar must be EXCLUDED as a feature whenever is_low_ctr is the label.")

AUC using ctr_mar ALONE to predict is_low_ctr: 1.000
-> this is expected to sit very close to 1.0, because is_low_ctr was DEFINED from ctr_mar.
   Conclusion: ctr_mar must be EXCLUDED as a feature whenever is_low_ctr is the label.


**Attack 2 — future-window leakage.** Every feature above should come from `month = '2026-03'` only. Confirming that no row secretly reaches into April or later, which would let a feature "see" outcomes that haven't happened yet relative to that window.

In [5]:
window_check = con.sql(f"""
    SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print('=== ATTACK 2: FUTURE-WINDOW CHECK ===')
print(window_check)
print("-> both dates should fall inside March 2026. If max_d ever crept into April, that would be leakage.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== ATTACK 2: FUTURE-WINDOW CHECK ===
       min_d      max_d
0 2026-03-01 2026-03-31
-> both dates should fall inside March 2026. If max_d ever crept into April, that would be leakage.


**Attack 3 — product-decision-flag leakage.** Confirming none of FlyRank's own rule-based outputs (`health_score`, `priority_score`, `action_type`, `refresh_tier`) exist in this table at all — so there's nothing to accidentally leak in as a feature, and nothing to remove.

In [6]:
schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 0").df()
banned_terms = ['health_score', 'priority_score', 'action_type', 'refresh_tier', 'flag']
hits = schema[schema['column_name'].str.contains('|'.join(banned_terms), case=False)]
print('=== ATTACK 3: PRODUCT-FLAG CHECK ===')
print(f'columns matching product-decision-flag names: {len(hits)}')
print(hits if len(hits) else '-> none found. Confirms the table ships observable signals only, as the lane guide states.')

=== ATTACK 3: PRODUCT-FLAG CHECK ===
columns matching product-decision-flag names: 0
-> none found. Confirms the table ships observable signals only, as the lane guide states.


## 4. What I excluded and why

| Excluded field | Why |
|---|---|
| `client_hash_id`, `content_hash_id` | Context/join keys only — used for grouping, never fed to a model as a feature |
| `report_date`, `month` | Used only to define the window, not as a feature — a raw calendar value risks the model memorizing a specific date rather than learning a generalizable pattern; a proper seasonality feature (e.g. month-of-year, cyclically encoded) would be engineered separately if needed, not the raw string |
| `ctr_mar` (conditionally) | Excluded specifically whenever `is_low_ctr` is the label — see Attack 1. Not excluded outright, since it's a legitimate feature for a *different* label (e.g. an engagement-only target) |
| Individual `ai_chatgpt` / `ai_perplexity` / `ai_gemini` / `ai_copilot` / `ai_claude` / `ai_meta` / `ai_other` columns | Excluded as separate features — the lane guide's own density numbers show AI-referral data is extremely sparse (30,177 rows out of ~79M), so six near-empty individual columns would mostly encode noise; collapsed into the single `sess_ai` total and the `primary_channel_mar` bucket instead |
| `health_score`, `priority_score`, `action_type`, `refresh_tier` | Confirmed absent from this table by Attack 3 — excluded on principle regardless, since using a rebuilt product decision as a feature would let a model just copy FlyRank's existing rule rather than discover its own signal |
| Any `dim_content` field (content type, word count, intent) | Not excluded permanently — just not included **yet**, because I haven't verified that table's real column names the way I verified `fact_content_daily_performance` this week. Guessing column names cost me a failed run on the data-contract notebook already; I'd rather confirm the schema first than repeat that mistake here |
| Raw query/URL/title text | Never in the release at all — scrambled before publication, so there's nothing to exclude, only to never attempt to reconstruct |

In [7]:
# No additional query needed here -- the table above is the deliverable for this section.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this in Colab with your HF_TOKEN before checking this box**
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.